# 가위와 바위를 모델은 어떻게 구분할까?

이 파일은 **읽기 전용 결과 설명**입니다. 실행할 코드 셀이 없습니다. 원본 데이터와 기존 분류·whitening 코드는 바꾸지 않았습니다.

기존 그림은 전극 5개를 평균해서 차이가 가려질 수 있었습니다. 여기서는 모델이 실제로 받은 **60개 전극 × 8개 시간 구간 = 480개 특징**을 펼칩니다. 모든 그림은 whitening 조건입니다.

## 1. 평균으로 합치지 않은 전극·시간 패턴

![전극과 시간별 패턴](results/team_figures/04_channel_time_patterns.png)

- **가로:** 지시 후 0.25–2.25초, 한 칸은 0.25초입니다.
- **세로:** ECoG 전극 1–60번입니다. 실제 뇌의 해부학적 배치 지도는 아닙니다.
- **A·B:** 모델을 학습시킨 바위 27회와 가위 27회의 특징 평균입니다. 각 칸의 색은 해당 특징의 학습 자료 전체 81회 평균·표준편차로 표준화한 값입니다. 주황색은 학습 평균보다 높은 파워, 파란색은 낮은 파워입니다.
- **C:** 가위 평균에서 바위 평균을 뺀 값입니다. 주황색 칸은 가위가 더 높고, 파란색 칸은 바위가 더 높습니다. 흰색에 가까우면 평균 차이가 작습니다. C의 색상 척도는 A·B와 별도입니다.

**이 색은 동작 전 대비 dB가 아닙니다.** 전극마다 원래 파워 크기가 달라서, 학습 자료 기준 표준편차 단위로 바꾸어 비교했습니다. 표준화는 표시용이며 분류기에 새로 적용하지 않았습니다. 오분류한 65번의 교차검증 fold를 고정해 학습 81회만으로 이 기준을 만들었습니다. 테스트 9회는 기준 계산과 학습에서 제외했습니다.

## 2. 같은 모델의 실제 테스트 판정

![실제 판정과 특징별 기여](results/team_figures/05_held_out_decisions.png)

왼쪽은 **44번: 바위 → 바위**, 가운데는 **8번: 가위 → 가위**, 오른쪽은 **65번: 바위 → 가위로 오분류**입니다. 세 사례 모두 같은 fold의 테스트 자료이며, 이 모델은 이 세 사례로 학습하지 않았습니다.

위쪽 지도는 각 특징이 **가위 점수 − 바위 점수**에 더한 값을 보여줍니다. 주황색은 가위 쪽으로, 파란색은 바위 쪽으로 점수를 밀어 줍니다. 480개 칸을 모두 더한 뒤 그림에 표시한 고정 offset을 더하면 실제 점수 차이가 됩니다.

아래 막대는 세 동작의 모델 점수입니다. 읽기 편하게 바위 점수를 모두에서 뺀 값으로 표시했으며, 이렇게 해도 가장 높은 동작은 바뀌지 않습니다. **점수는 확률이나 정확도가 아닙니다.** 가장 높은 막대의 동작을 예측합니다. 오른쪽 사례는 바위 지시였지만 가위 점수가 더 높아 틀렸습니다.

모델은 평균 곡선에 가장 가까운 동작을 찾는 것이 아니라, 학습한 전극·시간별 가중치로 480개 특징을 합쳐 점수를 계산합니다. 따라서 전체 곡선이 비슷해도 판정이 달라질 수 있습니다.

## 해석 범위와 검증

첫 번째 10-fold 평가의 **10개 모델·90개 테스트 예측을 같은 설정으로 재구성**했고, 저장된 예측과 전부 일치했습니다. 기존 정확도를 높이기 위한 재설정이나 새 모델 선택은 하지 않았습니다.

특징별 기여도는 이 LDA가 계산한 점수를 설명합니다. 전극이 특정 손가락만 담당한다거나, 그 전극이 움직임의 원인이라는 뜻은 아닙니다. 전극 간 상관이 있어 칸 하나의 기여를 독립적인 생물학적 중요도로 읽지 않습니다. 동작 정답은 화면의 지시 라벨이며 실제 손 자세의 독립 검증값이 아닙니다.

수치와 선택 기준: `results/team_figures/decoding_metadata.json`

실제 지도·점수 배열: `results/team_figures/decoding_figure_data.npz`